# ✈️ Notebook 2: Data Preprocessing & Feature Engineering
**Project:** Flight Delay Prediction  
**Phase 1:** Traditional Machine Learning Only (Academic Syllabus)  

---

## 🎯 Objectives:
1. **Operational Filtering:** Filter out cancelled flights, diverted flights, and missing delay entries with full row tracking.
2. **Binary Target Formulation:** Create $y = 1$ if `arrival_delay >= 15` min, $0$ otherwise.
3. **Chronological Splitting:** Partition data into Train (70%), Validation (15%), and Test (15%) with **zero date overlap**.
4. **Feature Engineering:**
   - Scheduled departure hour and minute.
   - Cyclic sine/cosine encoding for periodic time-of-day features.
   - Calendar features: day of week, weekend flag, month.
   - Route combination (`ORIGIN_DEST`).
5. **Preprocessing Pipeline (scikit-learn):**
   - Missing value imputation (`SimpleImputer`).
   - Feature scaling (`StandardScaler`) — essential for Gradient Descent & Regularization.
   - One-hot encoding (`OneHotEncoder(handle_unknown='ignore')`).
6. **Integrity Assertions:** Verify zero target leakage and zero date overlap.


In [1]:
# Step 1: Imports
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(".."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from src.config import (
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    TARGET_COLUMN,
    LEAKAGE_COLUMNS,
    RANDOM_STATE
)
from src.data_loader import find_dataset_files, detect_column_mapping
from src.preprocessing import (
    prepare_classification_dataset,
    chronological_train_val_test_split,
    engineer_pre_departure_features,
    build_column_transformer,
    assert_no_target_leakage,
    assert_chronological_integrity,
    assert_no_null_targets,
    generate_diagnostic_flight_data
)

print("✓ All preprocessing tools loaded successfully.")


✓ All preprocessing tools loaded successfully.


In [2]:
# Step 2: Ingest dataset or initialize benchmark test data
dataset_files = find_dataset_files()

if dataset_files:
    primary_file = dataset_files[0]
    print(f"✓ Ingesting real dataset: {primary_file.name}")
    file_size_mb = primary_file.stat().st_size / (1024 * 1024)
    sample_limit = 200_000 if file_size_mb > 200 else None
    raw_df = pd.read_csv(primary_file, nrows=sample_limit, low_memory=False)
    mapping, _ = detect_column_mapping(raw_df.columns)
else:
    print("ℹ️  Live CSV not detected in data/raw/. Generating diagnostic flight dataset (5,000 flights)")
    print("   to execute and verify all preprocessing transformations.")
    raw_df, mapping = generate_diagnostic_flight_data(5000, random_state=RANDOM_STATE)

print(f"Ingested shape: {raw_df.shape[0]:,} rows, {raw_df.shape[1]} columns.")


ℹ️  Live CSV not detected in data/raw/. Generating diagnostic flight dataset (5,000 flights)
   to execute and verify all preprocessing transformations.
Ingested shape: 5,000 rows, 20 columns.


In [3]:
# Step 3: Operational Filtering and Target Creation
clean_df, audit = prepare_classification_dataset(
    raw_df,
    mapping,
    filter_cancelled=True,
    filter_diverted=True
)

print("=" * 65)
print("OPERATIONAL FILTERING AUDIT REPORT")
print("=" * 65)
print(f"Initial Records              : {audit['initial_rows']:,}")
print(f"Cancelled Flights Excluded   : -{audit['cancelled_rows_removed']:,}")
print(f"Diverted Flights Excluded    : -{audit['diverted_rows_removed']:,}")
print(f"Missing Target Excluded      : -{audit['missing_target_rows_removed']:,}")
print("-" * 65)
print(f"Clean Operational Records    : {audit['final_rows']:,} ({audit['retention_pct']:.2f}% retained)")
print(f"Delayed (>=15m) Target Rate  : {audit['target_positive_pct']:.2f}%")
print("=" * 65)


OPERATIONAL FILTERING AUDIT REPORT
Initial Records              : 5,000
Cancelled Flights Excluded   : -92
Diverted Flights Excluded    : -17
Missing Target Excluded      : -0
-----------------------------------------------------------------
Clean Operational Records    : 4,891 (97.82% retained)
Delayed (>=15m) Target Rate  : 40.67%


In [4]:
# Step 4: Chronological Train / Validation / Test Split
train_df, val_df, test_df, split = chronological_train_val_test_split(
    clean_df,
    mapping,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

print("=" * 70)
print("CHRONOLOGICAL PARTITIONS (ZERO DATE OVERLAP)")
print("=" * 70)
print(f"Train : {split['train_rows']:,} rows | {split['train_start_date']} to {split['train_end_date']} | Delay: {split['train_delay_rate_pct']:.2f}%")
print(f"Val   : {split['val_rows']:,} rows | {split['val_start_date']} to {split['val_end_date']} | Delay: {split['val_delay_rate_pct']:.2f}%")
print(f"Test  : {split['test_rows']:,} rows | {split['test_start_date']} to {split['test_end_date']} | Delay: {split['test_delay_rate_pct']:.2f}%")
print("=" * 70)


CHRONOLOGICAL PARTITIONS (ZERO DATE OVERLAP)
Train : 3,408 rows | 2022-01-01 to 2022-03-03 | Delay: 40.17%
Val   : 765 rows | 2022-03-04 to 2022-03-17 | Delay: 42.61%
Test  : 718 rows | 2022-03-18 to 2022-03-31 | Delay: 40.95%


In [5]:
# Step 5: Pre-Departure Feature Engineering
X_train, y_train = engineer_pre_departure_features(train_df, mapping)
X_val, y_val = engineer_pre_departure_features(val_df, mapping)
X_test, y_test = engineer_pre_departure_features(test_df, mapping)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print("\nEngineered pre-departure features:", list(X_train.columns))


X_train shape: (3408, 19)
y_train shape: (3408,)

Engineered pre-departure features: ['departure_hour', 'departure_minute', 'dep_hour_sin', 'dep_hour_cos', 'dep_minute_of_day_sin', 'dep_minute_of_day_cos', 'day_of_week', 'is_weekend', 'day_of_week_sin', 'day_of_week_cos', 'month', 'month_sin', 'month_cos', 'distance', 'scheduled_elapsed_time', 'airline', 'origin', 'destination', 'route']


In [6]:
# Step 6: Scikit-Learn ColumnTransformer and Pipeline
numeric_features = list(X_train.select_dtypes(include=["number"]).columns)
categorical_features = ["airline", "origin", "destination"]

print(f"Numerical features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

# Build ColumnTransformer (StandardScaler + OneHotEncoder)
preprocessor = build_column_transformer(numeric_features, categorical_features, scale_numeric=True)

# Fit strictly on X_train
preprocessor.fit(X_train)

# Transform train, val, test
X_train_transformed = preprocessor.transform(X_train)
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)

print("\n✓ Preprocessing complete:")
print(f"  X_train_transformed shape: {X_train_transformed.shape}")
print(f"  X_val_transformed shape  : {X_val_transformed.shape}")
print(f"  X_test_transformed shape : {X_test_transformed.shape}")


Numerical features (15): ['departure_hour', 'departure_minute', 'dep_hour_sin', 'dep_hour_cos', 'dep_minute_of_day_sin', 'dep_minute_of_day_cos', 'day_of_week', 'is_weekend', 'day_of_week_sin', 'day_of_week_cos', 'month', 'month_sin', 'month_cos', 'distance', 'scheduled_elapsed_time']
Categorical features (3): ['airline', 'origin', 'destination']

✓ Preprocessing complete:
  X_train_transformed shape: (3408, 42)
  X_val_transformed shape  : (765, 42)
  X_test_transformed shape : (718, 42)


In [7]:
# Step 7: Integrity and Anti-Leakage Assertions
assert_no_target_leakage(X_train.columns, LEAKAGE_COLUMNS, TARGET_COLUMN)
assert_no_null_targets(y_train, y_val, y_test)
assert X_train_transformed.shape[1] == X_val_transformed.shape[1] == X_test_transformed.shape[1]
print("✓ All data integrity and anti-leakage assertions passed successfully.")


✓ All data integrity and anti-leakage assertions passed successfully.


## 📝 Notebook 2 Summary & Next Steps
- Excluded cancelled/diverted records with full audit transparency.
- Formulated binary classification target $y \in \{0, 1\}$.
- Established chronological train/val/test splits without date overlap.
- Applied `StandardScaler` and `OneHotEncoder` fitted exclusively on training data.
- **Next Notebook:** `03_logistic_regression.ipynb` will introduce Logistic Regression and Maximum Likelihood Estimation.
